In [1]:
import os
import sys
import glob

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)
    
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from pycaret.clustering import setup, create_model, assign_model, models, pull
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from datetime import datetime
import warnings
import time

from utils import preprocessing

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\tj\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [2]:
warnings.filterwarnings('ignore')
DATA_PATH = '../data'
OUTPUT_DIR = '../results'
IMAGE_DIR = '../images'
    
document_df = preprocessing.get_default_data()

📂 51개 파일 발견
🔄 텍스트 전처리 시작...
   옵션: HTML제거=True, URL제거=True, 숫자제거=True
   옵션: 불용어제거=True, Lemmatization=True, Stemming=False
✅ 전처리 완료:
   - 원본 문서 수: 51
   - 제거된 빈 문서: 0
   - 최종 문서 수: 51
   - 평균 단어 수: 1266.9


In [ ]:
# 확장 함수(yjh_00similarity 참고) 코드 : 
# 대표문서 -> 평균 벡터와 가장 가까운 문서

from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.metrics.pairwise import cosine_similarity

def auto_kmeans_with_representatives(document_df, text_column='processed_text',
                                     max_features=5000, svd_dim=50, k_range=(2,10), top_n=5):
    """
    document_df만 넣으면 자동으로:
    1. TF-IDF 벡터화
    2. SVD 차원 축소
    3. KMeans 여러 k 평가 (Silhouette, Davies-Bouldin, Calinski-Harabasz)
    4. 최적 k 선택
    5. 클러스터별 대표 문서(평균 벡터와 가장 가까운 문서) 및 유사 문서 Top-N 추출
    """
    # 1. TF-IDF 벡터화
    texts = document_df[text_column]
    vectorizer = TfidfVectorizer(max_features=max_features, stop_words='english')
    X = vectorizer.fit_transform(texts)

    # 2. 차원 축소 (SVD)
    svd = TruncatedSVD(n_components=svd_dim, random_state=42)
    X_reduced = svd.fit_transform(X)

    # 3. 여러 k 값 평가
    scores = {}
    for k in range(k_range[0], k_range[1] + 1):
        kmeans = KMeans(n_clusters=k, random_state=42)
        labels = kmeans.fit_predict(X_reduced)
        
        if len(set(labels)) > 1:
            sil = silhouette_score(X_reduced, labels)
            db = davies_bouldin_score(X_reduced, labels)
            ch = calinski_harabasz_score(X_reduced, labels)
            scores[k] = {"silhouette": sil, "davies_bouldin": db, "calinski": ch}
        else:
            scores[k] = {"silhouette": None, "davies_bouldin": None, "calinski": None}

    # 4. 지표 정규화 및 최적 k 선택
    sil_values = [v["silhouette"] for v in scores.values() if v["silhouette"] is not None]
    db_values  = [v["davies_bouldin"] for v in scores.values() if v["davies_bouldin"] is not None]
    ch_values  = [v["calinski"] for v in scores.values() if v["calinski"] is not None]

    def normalize(values, higher_is_better=True):
        arr = np.array(values)
        if higher_is_better:
            return (arr - arr.min()) / (arr.max() - arr.min())
        else:
            return (arr.max() - arr) / (arr.max() - arr.min())

    sil_norm = normalize(sil_values, higher_is_better=True)
    db_norm  = normalize(db_values, higher_is_better=False)
    ch_norm  = normalize(ch_values, higher_is_better=True)

    combined_scores = sil_norm + db_norm + ch_norm
    valid_keys = [k for k,v in scores.items() if v["silhouette"] is not None]
    best_index = np.argmax(combined_scores)
    best_k = valid_keys[best_index]

    # 5. 최적 k로 KMeans 실행
    final_kmeans = KMeans(n_clusters=best_k, random_state=42)
    labels = final_kmeans.fit_predict(X_reduced)
    document_df['Cluster'] = labels

    # 6. 클러스터별 대표 문서 (평균 벡터와 가장 가까운 문서) 및 유사 문서 Top-N 추출
    similarity_matrix = cosine_similarity(X)
    representatives = {}

    for cluster_id in range(best_k):
        cluster_docs = document_df[document_df['Cluster'] == cluster_id]
        if cluster_docs.empty:
            continue
        
        # 클러스터 평균 벡터 계산
        cluster_indices = cluster_docs.index
        cluster_vectors = X[cluster_indices].toarray()
        cluster_center = cluster_vectors.mean(axis=0)

        # 평균 벡터와 가장 가까운 문서 찾기
        sims_to_center = cosine_similarity([cluster_center], cluster_vectors)[0]
        rep_idx_local = np.argmax(sims_to_center)
        rep_idx = cluster_indices[rep_idx_local]
        rep_text = document_df.loc[rep_idx, text_column]

        # 대표 문서와 유사 문서 Top-N
        sim_scores = similarity_matrix[rep_idx]
        top_indices = np.argsort(sim_scores)[-top_n-1:-1]  # 자기 자신 제외
        similar_docs = document_df.iloc[top_indices][[text_column, 'Cluster']]
        
        representatives[cluster_id] = {
            "representative": rep_text,
            "similar_docs": similar_docs
        }

    return {
        "scores": scores,
        "best_k": best_k,
        "best_metrics": scores[best_k],
        "combined_score": combined_scores[best_index],
        "representatives": representatives
    }

In [ ]:
result = auto_kmeans_with_representatives(document_df, text_column='processed_text', top_n=5)

print("=== 최적 k 선택 결과 ===")
print(f"Best k = {result['best_k']}")
print(f"Silhouette = {result['best_metrics']['silhouette']:.3f}")
print(f"Davies-Bouldin = {result['best_metrics']['davies_bouldin']:.3f}")
print(f"Calinski-Harabasz = {result['best_metrics']['calinski']:.3f}")
print(f"Combined Score = {result['combined_score']:.3f}")

print("\n=== 클러스터별 대표 문서 및 유사 문서 ===")
for cluster_id, info in result['representatives'].items():
    print(f"\nCluster {cluster_id}:")
    print("대표 문서:", info['representative'])
    print("유사 문서 Top-N:")
    print(info['similar_docs'])
    
'''
- 이제 대표 문서는 클러스터 평균 벡터와 가장 가까운 문서로 선택
- 따라서 각 클러스터의 “중심”을 잘 대표하는 문서를 뽑을 수 있음
- 유사 문서 Top-N 도 함께 제공, 클러스터별 대표성과 다양성을 동시에 확인
'''

=== 최적 k 선택 결과 ===
Best k = 10
Silhouette = 0.152
Davies-Bouldin = 1.789
Calinski-Harabasz = 2.764
Combined Score = 2.000

=== 클러스터별 대표 문서 및 유사 문서 ===

Cluster 0:
대표 문서: thing like point must push micro sized right angle end ac adapter snap place battery may charge full size right shift key enough drop understand full featured laptop squeezed smaller manageable size keyboard size found standard computer machine run little hot keyboard responsive feel larger key along oversize right shift key real plus yes airplane trip last couple day thankful small size person front reclined seat back way whole flight taking machine airport security much easier full size notebook even battery life continues hold keyboard true godsend full size key recommend minimizing size scroll zone right bottom wise find screen suddenly scrolling want huge improvement old full size laptop many way thing less ideal big deal tiny light slightly smaller standard keyboard problem use touch typist key good size feel goo

In [5]:
# PCA/t-SNE 시각화와 결합
# 각 클러스터별 대표 문서를 그래프 상에 강조 표시

def visualize_clusters_with_representatives(document_df, X_reduced, labels, representatives, method='pca'):
    """
    클러스터링 결과를 PCA/t-SNE로 시각화하고 대표 문서를 강조 표시
    
    Parameters:
        document_df (pd.DataFrame): 원본 데이터프레임
        X_reduced (ndarray): 차원 축소된 벡터
        labels (array): KMeans 클러스터 라벨
        representatives (dict): 클러스터별 대표 문서 정보
        method (str): 'pca' 또는 'tsne'
    """
    # 차원 축소
    if method == 'pca':
        reducer = PCA(n_components=2, random_state=42)
        coords = reducer.fit_transform(X_reduced)
    elif method == 'tsne':
        reducer = TSNE(n_components=2, random_state=42, perplexity=30)
        coords = reducer.fit_transform(X_reduced)
    else:
        raise ValueError("method는 'pca' 또는 'tsne'만 가능합니다.")

    # 시각화
    plt.figure(figsize=(10,8))
    scatter = plt.scatter(coords[:,0], coords[:,1], c=labels, cmap='tab10', alpha=0.6)

    # 대표 문서 강조 표시
    for cluster_id, info in representatives.items():
        rep_text = info['representative']
        # 대표 문서 인덱스 찾기
        rep_idx = document_df[document_df['processed_text'] == rep_text].index[0]
        plt.scatter(coords[rep_idx,0], coords[rep_idx,1],
                    marker='*', s=250, color='black', edgecolor='white',
                    label=f"Cluster {cluster_id} 대표")

    plt.title(f"문서 군집화 시각화 ({method.upper()})")
    plt.xlabel("Component 1")
    plt.ylabel("Component 2")
    plt.legend()
    plt.grid(True)
    plt.show()

In [6]:
# auto_kmeans_with_representatives 실행 결과 활용
result = auto_kmeans_with_representatives(document_df, text_column='processed_text', top_n=5)

labels = document_df['Cluster'].values
representatives = result['representatives']

# PCA 시각화
visualize_clusters_with_representatives(document_df, X_reduced, labels, representatives, method='pca')

# t-SNE 시각화
visualize_clusters_with_representatives(document_df, X_reduced, labels, representatives, method='tsne')

NameError: name 'X_reduced' is not defined